# 06 - Modelos clásicos

## 1. Configuración y carga

In [1]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
FEATURES = ["brillo", "contraste", "saturacion", "green_ratio", "edge_density"]
CLASES = ["bg", "challenge", "play", "throwin"]

MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

frames = pd.read_csv("../data/processed/frame_metrics.csv")
X, y, groups = frames[FEATURES], frames["label"], frames["video_id"]
print(f"{len(frames)} frames de {groups.nunique()} videos")

9133 frames de 3 videos


## 2. Cómo se evalúa

Solo hay frames de 3 partidos. Cada modelo se entrena 3 veces, dejando cada vez un partido
completo fuera para probar (*leave-one-video-out*). Así el modelo nunca ve en prueba un partido
que ya vio entrenando.

In [2]:
def evaluar(nombre, pipeline):
    """Evalúa con leave-one-video-out, guarda el modelo final y devuelve (métricas, predicciones)."""
    filas_metricas, filas_preds = [], []

    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
        modelo = clone(pipeline)
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_te, y_te = X.iloc[test_idx], y.iloc[test_idx]

        t0 = time.perf_counter()
        modelo.fit(X_tr, y_tr)
        train_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        pred = modelo.predict(X_te)
        proba_raw = modelo.predict_proba(X_te)
        pred_time_ms = (time.perf_counter() - t0) * 1000 / len(X_te)

        # columnas de probabilidad siempre en el orden de CLASES
        proba = pd.DataFrame(0.0, index=X_te.index, columns=CLASES)
        proba[list(modelo.classes_)] = proba_raw

        f1_clase = f1_score(y_te, pred, labels=CLASES, average=None, zero_division=0)
        filas_metricas.append({
            "model": nombre,
            "fold": groups.iloc[test_idx].iloc[0],
            "accuracy": accuracy_score(y_te, pred),
            "f1_macro": f1_score(y_te, pred, labels=CLASES, average="macro", zero_division=0),
            **{f"f1_{c}": f for c, f in zip(CLASES, f1_clase)},
            "train_time_s": train_time,
            "pred_time_ms": pred_time_ms,
        })

        filas_preds.append(pd.DataFrame({
            "video_id": frames["video_id"].iloc[test_idx].values,
            "time_s": frames["time_s"].iloc[test_idx].values,
            "label": y_te.values,
            "model": nombre,
            "pred": pred,
            **{f"proba_{c}": proba[c].values for c in CLASES},
        }))

    # modelo final con todos los datos, es el que usa la app
    final = clone(pipeline).fit(X, y)
    joblib.dump(final, MODELS_DIR / f"{nombre}.joblib", compress=3)

    metricas = pd.DataFrame(filas_metricas)[[
        "model", "fold", "accuracy", "f1_macro", "f1_bg", "f1_play",
        "f1_challenge", "f1_throwin", "train_time_s", "pred_time_ms",
    ]]
    return metricas, pd.concat(filas_preds, ignore_index=True)

## 3. Modelo de referencia (Dummy)

Siempre predice la clase más común (`play`). Cualquier modelo real tiene que superarlo.

In [3]:
resultados = {}

resultados["dummy"] = evaluar("dummy", Pipeline([
    ("clf", DummyClassifier(strategy="most_frequent")),
]))
resultados["dummy"][0]

,model,fold,accuracy,f1_macro,f1_bg,f1_play,f1_challenge,f1_throwin,train_time_s,pred_time_ms
0,dummy,1606b0e6_0,0.733678,0.211596,0.0,0.846383,0.0,0.0,0.001152,0.000036
1,dummy,1606b0e6_1,0.801492,0.222452,0.0,0.889809,0.0,0.0,0.000857,0.000020
2,dummy,35bd9041_0,0.803474,0.222757,0.0,0.891029,0.0,0.0,0.000932,0.000022


## 4. Random Forest (mismos parámetros del baseline)

In [4]:
resultados["rf"] = evaluar("rf", Pipeline([
    ("clf", RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
]))
resultados["rf"][0]

,model,fold,accuracy,f1_macro,f1_bg,f1_play,f1_challenge,f1_throwin,train_time_s,pred_time_ms
0,rf,1606b0e6_0,0.637772,0.224416,0.103560,0.794104,0.000000,0.0,0.299773,0.018746
1,rf,1606b0e6_1,0.416236,0.214653,0.153022,0.581967,0.123624,0.0,0.286781,0.014829
2,rf,35bd9041_0,0.751703,0.243826,0.038251,0.864886,0.072165,0.0,0.301262,0.017438


## 5. Guardar resultados

In [5]:
metrics = pd.concat([m for m, _ in resultados.values()], ignore_index=True)
preds = pd.concat([p for _, p in resultados.values()], ignore_index=True)

metrics.to_csv(RESULTS_DIR / "metrics_clasicos.csv", index=False)
preds.to_csv(RESULTS_DIR / "predictions_clasicos.csv", index=False)
print(f"metrics: {metrics.shape}, predictions: {preds.shape}")

metrics: (6, 10), predictions: (18266, 9)


## 6. Resumen (promedio ± desviación de los 3 partidos)

In [6]:
cols = ["accuracy", "f1_macro", "f1_bg", "f1_play", "f1_challenge", "f1_throwin"]
resumen = metrics.groupby("model")[cols].agg(["mean", "std"]).round(3)
resumen

accuracy        f1_macro         f1_bg        f1_play         \
          mean    std     mean    std   mean    std    mean    std   
model                                                                
dummy    0.780  0.040    0.219  0.006  0.000  0.000   0.876  0.025   
rf       0.602  0.171    0.228  0.015  0.098  0.058   0.747  0.147   

      f1_challenge        f1_throwin       
              mean    std       mean  std  
model                                      
dummy        0.000  0.000        0.0  0.0  
rf           0.065  0.062        0.0  0.0

El Random Forest apenas se separa del Dummy: con 5 propiedades globales de la imagen casi no
logra reconocer `challenge` ni `throwin`. Lo siguiente es ajustar parámetros y probar otros
modelos.